# Project 6: Data Exploration and Visualization

**Student work – CS110 W26**

## Part 1: Dataset Selection and Initial Exploration

### 1. Dataset Selection

**Dataset:** *Restaurant Tips* – a 244-row, 7-column dataset recording every table served over several weeks at a single restaurant.  
**Columns:** `total_bill` (float), `tip` (float), `sex` (category), `smoker` (category), `day` (category), `time` (category), `size` (int).  
The dataset meets the requirements: >100 rows, mixed numerical and categorical types, and is self-contained (CSV included with submission).

### 2. Import Libraries

In [ ]:
# Standard data-science imports
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# Nicer default aesthetics
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

print('Libraries imported successfully.')

### 3. Load the Dataset

In [ ]:
# Load the CSV that is included with this submission
df = pd.read_csv('tips.csv')

# Convert categorical columns to the proper dtype for memory efficiency
for col in ['sex', 'smoker', 'day', 'time']:
    df[col] = df[col].astype('category')

print('Dataset loaded. Shape:', df.shape)

### 4. Data Summary

In [ ]:
# 4a – Dataset structure
df.info()

In [ ]:
# 4b – Statistical summary of numerical columns
df.describe()

In [ ]:
# 4c – First 5 rows
df.head()

In [ ]:
# 4d – Dimensions (rows, columns)
print('Shape:', df.shape)

In [ ]:
# 4e – Column names
print('Columns:', df.columns.tolist())

### 5. Inquiry Question

**Does the day of the week and party size influence the tip percentage customers leave, and do smoking status or gender play a meaningful role?**

Understanding what drives tip percentage helps restaurant management schedule staff and design seating policies.

## Part 2: Data Visualization and Analysis

### 1. Data Cleaning

In [ ]:
# Check for missing values
print('Missing values per column:')
print(df.isnull().sum())

# Check for duplicate rows
print(f'\nDuplicate rows: {df.duplicated().sum()}')

# The dataset is already clean – no action needed
print('\nNo cleaning required.')

In [ ]:
# Engineer a 'tip_pct' column (tip as % of total bill) – our key metric
df['tip_pct'] = (df['tip'] / df['total_bill'] * 100).round(2)

print('tip_pct column added. Sample values:')
print(df[['total_bill', 'tip', 'tip_pct']].head())

### 2 & 3. Visualizations

Four plots are created below, each building toward answering the inquiry question.

In [ ]:
# ── Visualization 1: Tip percentage by day of the week (box plot) ──

# Order days logically
day_order = ['Thur', 'Fri', 'Sat', 'Sun']

fig, ax = plt.subplots(figsize=(8, 5))

sns.boxplot(
    data=df, x='day', y='tip_pct',
    order=day_order, palette='Blues',
    flierprops=dict(marker='o', markersize=4, alpha=0.5),
    ax=ax
)

ax.set_title('Tip Percentage by Day of the Week', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Day', fontsize=12)
ax.set_ylabel('Tip Percentage (%)', fontsize=12)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f%%'))

# Annotate median values
for i, day in enumerate(day_order):
    med = df[df['day'] == day]['tip_pct'].median()
    ax.text(i, med + 0.8, f'{med:.1f}%', ha='center', fontsize=9, color='navy')

plt.tight_layout()
plt.savefig('viz1_tip_pct_by_day.png', bbox_inches='tight')
plt.show()
print('Figure saved as viz1_tip_pct_by_day.png')

In [ ]:
# ── Visualization 2: Total bill vs tip amount, colored by smoker status ──

fig, ax = plt.subplots(figsize=(8, 5))

for label, grp in df.groupby('smoker'):
    ax.scatter(
        grp['total_bill'], grp['tip'],
        label=f'Smoker: {label}',
        alpha=0.65, edgecolors='white', linewidth=0.4, s=55
    )

ax.set_title('Total Bill vs Tip Amount by Smoker Status', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Total Bill ($)', fontsize=12)
ax.set_ylabel('Tip ($)', fontsize=12)
ax.legend(title='Smoker', fontsize=10)

# Add a thin reference line at the average tip rate (~16 %)
x_vals = [df['total_bill'].min(), df['total_bill'].max()]
ax.plot(x_vals, [v * 0.16 for v in x_vals],
        linestyle='--', color='gray', linewidth=1, label='16% reference')
ax.legend(title='', fontsize=9)

plt.tight_layout()
plt.savefig('viz2_bill_vs_tip_smoker.png', bbox_inches='tight')
plt.show()
print('Figure saved as viz2_bill_vs_tip_smoker.png')

In [ ]:
# ── Visualization 3: Average tip percentage by party size (bar chart) ──

avg_by_size = (
    df.groupby('size')['tip_pct']
    .agg(['mean', 'sem'])   # mean and standard error
    .reset_index()
)

fig, ax = plt.subplots(figsize=(7, 5))

bars = ax.bar(
    avg_by_size['size'].astype(str),
    avg_by_size['mean'],
    yerr=avg_by_size['sem'],
    capsize=5,
    color=sns.color_palette('Greens_d', len(avg_by_size)),
    edgecolor='white', linewidth=0.8
)

# Label each bar with its value
for bar, val in zip(bars, avg_by_size['mean']):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.3,
            f'{val:.1f}%', ha='center', va='bottom', fontsize=9)

ax.set_title('Average Tip Percentage by Party Size\n(error bars = ±1 SE)', fontsize=13, fontweight='bold', pad=10)
ax.set_xlabel('Party Size (number of diners)', fontsize=12)
ax.set_ylabel('Average Tip Percentage (%)', fontsize=12)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f%%'))
ax.set_ylim(0, avg_by_size['mean'].max() + 3)

plt.tight_layout()
plt.savefig('viz3_tip_pct_by_size.png', bbox_inches='tight')
plt.show()
print('Figure saved as viz3_tip_pct_by_size.png')

In [ ]:
# ── Visualization 4: Mean tip percentage by sex and smoker status (grouped bar) ──

pivot = (
    df.groupby(['sex', 'smoker'])['tip_pct']
    .mean()
    .unstack('smoker')  # columns = Yes / No
)

fig, ax = plt.subplots(figsize=(7, 5))

pivot.plot(kind='bar', ax=ax, color=['#e07b54', '#5b8db8'],
           edgecolor='white', linewidth=0.8, rot=0)

ax.set_title('Mean Tip % by Gender and Smoker Status', fontsize=13, fontweight='bold', pad=10)
ax.set_xlabel('Gender', fontsize=12)
ax.set_ylabel('Mean Tip Percentage (%)', fontsize=12)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.1f%%'))
ax.legend(title='Smoker', labels=['Non-Smoker', 'Smoker'], fontsize=10)
ax.set_ylim(0, pivot.values.max() + 3)

# Annotate bars
for container in ax.containers:
    ax.bar_label(container, fmt='%.1f%%', padding=3, fontsize=9)

plt.tight_layout()
plt.savefig('viz4_tip_pct_gender_smoker.png', bbox_inches='tight')
plt.show()
print('Figure saved as viz4_tip_pct_gender_smoker.png')

### 4. Analysis

**Research question:** *Does the day of the week and party size influence the tip percentage customers leave, and do smoking status or gender play a meaningful role?*

The visualizations collectively provide a clear answer. **Day of the week has a modest effect on tipping behavior** (Visualization 1): Friday diners leave the highest median tip (~17%), while Sunday and Thursday tables tip slightly lower (~15–16%). The wide interquartile ranges on all days indicate that individual variability is larger than the day-level effect, so day alone is not a strong predictor.

**Party size shows a more consistent downward trend in tip percentage** (Visualization 3): solo diners and couples tip at roughly 17–18%, while tables of five or six tip closer to 14–15%. This is a well-documented phenomenon—large groups often expect an automatic gratuity and may collectively tip less per capita. Visualization 2 confirms that the absolute tip amount grows with the bill, but the percentage relationship with the dashed 16% reference line shows considerable scatter for high-bill tables. Visualization 4 reveals that **smoking status and gender have only minor effects** on tip percentage once other variables are accounted for; differences are well under one percentage point in most groups.

**Limitations:** The dataset covers a single restaurant over a limited period, so generalizing to other settings requires caution. With only 244 rows the cells for rare party sizes (1, 5, 6) have few observations, making those estimates noisy (reflected in large error bars). **Further investigation** could include multivariate regression to isolate each factor's contribution, or a time-series analysis if timestamp data were available to check for seasonal trends.